In [ ]:
# Imports, seed, and run config (model / paths / max sequence length)
import math, random
import numpy as np
import pandas as pd
import torch
import json
from datasets import Dataset
from transformers import (
    GPT2TokenizerFast, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MODEL_NAME = 'gpt2-medium'
MAX_LEN    = 256
OUT_DIR    = 'data/gpt2_medium_scidomain'
CSV_PATH   = "data/combined_acl_arxiv_with_citations_final.csv"

In [ ]:
# Load the fixed train/val/test id split
with open("train_val_test_split.json", "r") as f:
    split_dict = json.load(f)

train_ids = set(split_dict["train"])
val_ids   = set(split_dict["val"])
test_ids  = set(split_dict["test"])

In [ ]:
# Load CSV and split by year
df = pd.read_csv(CSV_PATH).dropna(subset=['title', 'abstract', 'publication_year'])
df['text'] = df['title'].str.strip() + ' ' + df['abstract'].str.strip()

train_texts = df.loc[df["project_id"].isin(train_ids), 'text'].to_numpy()
val_texts   = df.loc[df["project_id"].isin(val_ids), 'text'].to_numpy()
print(f'train: {len(train_texts):,}   val: {len(val_texts):,}')

train: 104,685   val: 20,129


In [ ]:
# Tokenizer + model
tok = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

def tok_fn(batch):
    return tok(batch['text'], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_dict({'text': train_texts}).map(tok_fn, batched=True, remove_columns=['text'])
val_ds   = Dataset.from_dict({'text': val_texts}).map(tok_fn,   batched=True, remove_columns=['text'])

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/104685 [00:00<?, ? examples/s]

Map:   0%|          | 0/20129 [00:00<?, ? examples/s]

In [ ]:
# Training config. BS 2 + grad_accum 16 = effective BS 32 (safe on a 16GB T4).
# If you have a bigger GPU you can raise per_device_train_batch_size.
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# Baseline val perplexity before any training
base = trainer.evaluate()
print(f'baseline val loss = {base["eval_loss"]:.4f}   PPL = {math.exp(base["eval_loss"]):.2f}')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


baseline val loss = 3.4781   PPL = 32.40


In [ ]:
# Fine-tune GPT-2 medium on the pre-cutoff corpus
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.931793,2.735523,0.004200
2,2.862871,2.701429,0.004200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=3272, training_loss=2.9633876502076686, metrics={'train_runtime': 14944.4662, 'train_samples_per_second': 14.01, 'train_steps_per_second': 0.219, 'total_flos': 9.253247768099635e+16, 'train_loss': 2.9633876502076686, 'epoch': 2.0})

In [ ]:
# Final val perplexity
final = trainer.evaluate()
print(f'final val loss    = {final["eval_loss"]:.4f}   PPL = {math.exp(final["eval_loss"]):.2f}')

# Memorization probe: sample 2000 train rows, compare PPL to val
probe_idx   = np.random.RandomState(SEED).choice(len(train_ds), size=2000, replace=False)
probe_ds    = train_ds.select(probe_idx.tolist())
probe       = trainer.evaluate(eval_dataset=probe_ds)
ratio       = math.exp(final['eval_loss']) / math.exp(probe['eval_loss'])
print(f'train-sample PPL  = {math.exp(probe["eval_loss"]):.2f}   val/train ratio = {ratio:.2f}')
print('  (ratio > 1.5 -> mild overfitting; drop epochs or raise weight_decay)' if ratio > 1.5
      else '  train/val gap looks healthy.')

final val loss    = 2.7014   PPL = 14.90
train-sample PPL  = 16.44   val/train ratio = 0.91
  train/val gap looks healthy.


In [ ]:
# Quick sanity battery (same idea as your frozen-GPT-2 checks)
device = next(model.parameters()).device
model.eval()
samples = {
    'coherent_nlp': 'We propose a transformer-based architecture for cross-lingual transfer, leveraging multilingual pretraining to bridge low-resource and high-resource languages.',
    'word_salad':   'Banana refrigerator quantum democracy zebra flux pineapple neutron sandwich algorithm fluorescent biscuit.',
    'repetitive':   'The model is good. ' * 8,
    'chherent+salad': 'We propose a transformer-based architecture for cross-lingual transfer, leveraging multilingual pretraining to bridge low-resource and high-resource languages.Banana refrigerator quantum democracy zebra flux pineapple neutron sandwich algorithm fluorescent biscuit.',
}
for name, text in samples.items():
    enc = tok(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        ce = model(**enc, labels=enc['input_ids']).loss.item()
    print(f'  [{name:14s}] CE={ce:.3f}  PPL={math.exp(ce):.2f}')

  [coherent_nlp  ] CE=2.294  PPL=9.91
  [word_salad    ] CE=9.267  PPL=10581.23
  [repetitive    ] CE=0.971  PPL=2.64
  [chherent+salad] CE=4.978  PPL=145.20


In [ ]:
# Save final model + tokenizer
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print(f'saved to {OUT_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to ./gpt2_medium_scidomain


In [ ]:
# Cross-entropy (Surprise) scorer under the fine-tuned model
def compute_entropy(text):
    with torch.no_grad():
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN
        )

        input_ids = inputs["input_ids"].to(device)

        outputs = model(
            input_ids=input_ids,
            labels=input_ids
        )

        loss = outputs.loss  # mean cross-entropy
        return loss.item()


In [ ]:
# Score every paper's title+abstract
df["entropy_score"] = df["text"].apply(compute_entropy)

In [ ]:
# Save per-paper entropy scores (feeds the Surprise axis)
df[["project_id","entropy_score"]].to_csv("data/entropy_scores_2.csv", index=False)
print("Entropy computation complete and saved.")